In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import time
import datetime
import requests
import tqdm

# from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By

In [2]:
# # 크롬 드라이버 경로 설정
# path = r"C:\Users\NT551_11TH\Downloads\chromedriver-win64\chromedriver.exe"

### API 호출

In [71]:
def parse_annual_fee(fee_str):
    """
    텍스트 형태의 연회비에서 숫자만 추출하는 함수
    우선순위: 1. 국내전용 -> 2. 해외겸용 -> 3. 일반 숫자 -> 4. 0
    """
    if not fee_str: # 데이터가 아예 없거나 None일 경우
        return 0
    
    # 1. '국내전용' 글자 뒤에 나오는 숫자(쉼표 포함)를 찾음
    domestic_match = re.search(r'국내전용[^0-9]*([0-9,]+)', fee_str)
    if domestic_match:
        return int(domestic_match.group(1).replace(',', '')) # 쉼표 제거 후 정수로 변환
    
    # 2. 국내전용이 없다면 '해외겸용' 글자 뒤에 나오는 숫자를 찾음
    overseas_match = re.search(r'해외겸용[^0-9]*([0-9,]+)', fee_str)
    if overseas_match:
        return int(overseas_match.group(1).replace(',', ''))
        
    # 3. (예외처리) '국내/해외' 구분 없이 "15,000원" 처럼 숫자만 덜렁 있는 경우
    any_num_match = re.search(r'([0-9,]+)[ ]*원', fee_str)
    if any_num_match:
        return int(any_num_match.group(1).replace(',', ''))
        
    # 4. 연회비가 없거나("연회비 없음") 위 조건에 다 안 맞으면 0 반환
    return 0

In [78]:
# 카드 단일 정보 수집 및 파싱 함수
def collect_cards_batch(card_id_list, desc="카드 데이터 수집"):
    """
    카드 ID 리스트를 입력받아 API를 호출하고 데이터를 일괄 수집합니다.
    반환값: (성공한 데이터 리스트, 실패한 카드ID 리스트)
    실패(404 등) 시: (False, 에러상태코드_또는_None)
    """
    card_data_list = []
    fail_list = []
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
        'Referer': 'https://www.card-gorilla.com/' 
    }
    for card_id in tqdm.tqdm(card_id_list, desc=desc):
        url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"

        try:
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                raw_data = response.json()
    
                # 연회비
                raw_fee_str = raw_data.get('annual_fee_basic', '')
                parsed_fee = parse_annual_fee(raw_fee_str)
    
                # 카테고리 추출
                category_list = []
                if 'key_benefit' in raw_data and raw_data['key_benefit']:
                    for benefit in raw_data['key_benefit']:
                        if 'cate' in benefit and 'name' in benefit['cate']:
                            cate_name = benefit['cate']['name']
                            if cate_name not in ['기타', '유의사항'] and cate_name not in category_list:
                                category_list.append(cate_name)
    
                # 주요혜택 추출
                top_tags_list = []
                if 'top_benefit' in raw_data and raw_data['top_benefit']:
                    for top in raw_data['top_benefit']:
                        if 'tags' in top and top['tags']:
                            top_tags_list.append(" ".join(top['tags']))
    
                # 발급가능여부
                is_discontinued = raw_data.get('is_discon', False)
                issuable_status = 'X' if is_discontinued else 'O'
    
                card_info = {
                    '카드번호': card_id,
                    '카드명': raw_data.get('name', '이름없음'),
                    '카드사': raw_data.get('corp', {}).get('name', '알수없음'),
                    '카드타입': raw_data.get('cate', '알수없음'),
                    '출시일': raw_data.get('release_dt', '알수없음'),
                    '연회비': parsed_fee,
                    '전월실적': raw_data.get('pre_month_money', 0),
                    '카테고리': ", ".join(category_list),
                    '주요혜택': ", ".join(top_tags_list),
                    '발급가능여부': issuable_status
                }
                card_data_list.append(card_info)
                
            elif response.status_code == 404:
                pass
            else:
                fail_list.append(card_id)
                
        except Exception as e:
            print(f"[오류] {card_id}번 카드 정보 요청 중 예외 발생: {e}")

        time.sleep(0.1)
        
    return card_data_list, fail_list

In [ ]:
id_list = range(1, 3000)

In [79]:
if __name__ == "__main__":
    # 1차 수집
    print("=== 1차 수집 시작 ===")
    first_success_data, first_fail_ids = collect_cards_batch(id_list, desc="1차 수집")
    print(f"1차 완료: 성공 {len(first_success_data)}건, 실패 {len(first_fail_ids)}건")

    # 2. 실패한 건이 있다면 재수집 실행
    if first_fail_ids:
        print("\n=== 정보 요청 실패 건 재수집 시작 ===")
        # 실패한 ID 리스트를 그대로 다시 함수에 던져줍니다!
        retry_success_data, final_fail_ids = collect_cards_batch(first_fail_ids, desc="재수집")
        
        # 1차 성공 데이터와 재수집 성공 데이터를 합침
        final_card_data = first_success_data + retry_success_data
        print(f"재수집 완료: 복구 성공 {len(retry_success_data)}건, 최종 실패 {len(final_fail_ids)}건")
    else:
        final_card_data = first_success_data
        
    print(f"\n✅ 총 {len(final_card_data)}건의 카드 데이터 수집이 완료되었습니다.")

=== 1차 수집 시작 ===


1차 수집: 100%|████████████████████████████████████████████████████████████████████| 2502/2502 [33:46<00:00,  1.23it/s]

1차 완료: 성공 2502건, 실패 0건

✅ 총 2502건의 카드 데이터 수집이 완료되었습니다.


In [33]:
df = pd.DataFrame(final_card_data)
df.head()

,카드번호,카드명,카드사,카드타입,출시일,연회비,전월실적,카테고리,주요혜택,발급가능여부
0,1,신한카드 Hi-Point,신한카드,CRD,,0,0,"쇼핑, 모든가맹점, 주유, 금융, 통신, 적립","전국가맹점 0.2~2.0% 적립, 특정가맹점 1~5% 적립, 에이치디현대오일뱅크 6...",X
1,2,신한카드 Love,신한카드,CRD,,0,200000,"백화점, 패밀리레스토랑, 주유소, 영화, 적립, 테마파크, 경기관람","백화점/할인점 5% 할인, 스타벅스 20% 할인, GS칼텍스 60원/L 할인",X
2,3,신한카드 Lady,신한카드,CRD,,0,300000,"쇼핑, 푸드, 영화, 주유소, 교육/육아, 테마파크, 통신, 경기관람","3대백화점 5% 할인, 패밀리레스토랑 20% 할인, 놀이공원 30~50% 할인",X
3,4,SK에너지 신한카드The you,신한카드,CRD,,7000,300000,"주유소, 대중교통, 무이자할부, 테마파크, 영화, 헤어, 경기관람, 여행사","SK주유소 100원/L 할인, 지하철,버스,택시 3~7% 할인, 놀이공원 30~50...",X
4,8,신한카드 The CLASSIC-Y,신한카드,CRD,,0,0,"선택형, 모든가맹점, 주유소, 생활, 면세점, 진에어, 공연/전시, 프리미엄","Gift Option 매년1회 선택이용, 마이신한포인트 0.7~5% 적립, GS칼텍...",O


In [38]:
df.sort_values(by='카드번호', inplace=True)
df.tail()

,카드번호,카드명,카드사,카드타입,출시일,연회비,전월실적,카테고리,주요혜택,발급가능여부
2813,2981,KB NEED Global 카드,KB국민카드,CRD,2026-04-30,30000,0,"해외, 국내가맹점","해외 가맹점 3.5% 할인, 국내 가맹점 0.5% 할인",O
2814,2982,현대카드 체크(포인트형),현대카드,CHK,2026-04-28,5000,0,"국내외가맹점, 적립","국내외 가맹점 0.5% 적립, 일반음식점, 배달 5% 적립, 편의점, 대중교통 5...",O
2815,2983,현대카드 체크(캐시백형),현대카드,CHK,2026-04-28,5000,0,"국내외가맹점, 적립","국내외 가맹점 0.3% 캐시백, 일반음식점, 배달 3% 캐시백, 편의점, 대중교통...",O
2816,2984,현대카드 체크(Apple Pay Rewards),현대카드,CHK,2026-04-28,5000,300000,간편결제,Apple Pay 이용 금액 10% 캐시백,O
2817,2985,현대카드 ZERO Up,현대카드,CRD,2026-06-16,30000,0,"적립, 프리미엄 서비스","국내외 가맹점 1.2% 적립, 온라인몰/대형마트 2.4% 적립, 교육/주유/이동통신...",O


In [39]:
df['카드사'].unique()

array(['신한카드', '삼성카드', '우리카드', '씨티카드', 'KB국민카드', 'NH농협카드', '롯데카드', '하나카드',
       'IBK기업은행', 'SC제일은행', '현대카드', 'SSGPAY. CARD', '카카오뱅크', '케이뱅크',
       '우체국', 'iM뱅크', 'BNK부산은행', '광주은행', 'Sh수협은행', '신협', '교보증권', '유안타증권',
       'KDB산업은행', 'SBI저축은행', 'MG새마을금고', '제주은행', '전북은행', '카카오페이',
       'BC 바로카드', '유진투자증권', 'SK증권', '미래에셋증권', 'NH투자증권', 'KB증권', 'DB금융투자',
       '한국투자증권', '토스뱅크', '엔에이치엔페이코', '코나카드', '아이오로라', '토스', '한패스', '차이',
       '핀크카드', '다날', '트래블월렛', 'KG 파이낸셜', '핀트', '현대백화점', 'BNK경남은행',
       '네이버페이', '머니트리', 'OK캐쉬백', '웰컴저축은행'], dtype=object)

In [40]:
df['카드사'].value_counts()

카드사
롯데카드            549
삼성카드            371
KB국민카드          286
우리카드            278
신한카드            252
현대카드            221
NH농협카드          201
하나카드            185
IBK기업은행         116
BNK부산은행          44
BC 바로카드          43
iM뱅크             41
MG새마을금고          31
우체국              26
광주은행             24
씨티카드             19
제주은행             14
신협               14
전북은행             14
케이뱅크             10
SC제일은행           10
Sh수협은행            7
카카오뱅크             5
코나카드              4
현대백화점             4
KB증권              4
SBI저축은행           4
유안타증권             3
교보증권              3
SSGPAY. CARD      3
OK캐쉬백             2
차이                2
한패스               2
토스뱅크              2
카카오페이             2
미래에셋증권            2
SK증권              2
유진투자증권            2
DB금융투자            2
한국투자증권            1
KG 파이낸셜           1
KDB산업은행           1
머니트리              1
네이버페이             1
BNK경남은행           1
핀트                1
다날                1
트래블월렛             1
핀크카드              1
토스              

In [41]:
len(df)

2819

In [42]:
card_comp_list = ['삼성카드', '신한카드', '현대카드', 'KB국민카드', '롯데카드', 
                  '우리카드', '하나카드', 'NH농협카드', 'IBK기업은행', 'BC 바로카드']

In [44]:
top_df = df.loc[df['카드사'].isin(card_comp_list)]
print(top_df['카드사'].unique())
print(len(top_df))

['신한카드' '삼성카드' '우리카드' 'KB국민카드' 'NH농협카드' '롯데카드' '하나카드' 'IBK기업은행' '현대카드'
 'BC 바로카드']
2502


In [46]:
top_df['카드사'].value_counts()

카드사
롯데카드       549
삼성카드       371
KB국민카드     286
우리카드       278
신한카드       252
현대카드       221
NH농협카드     201
하나카드       185
IBK기업은행    116
BC 바로카드     43
Name: count, dtype: int64

In [51]:
top_df.to_csv('data/card_data.csv', index=False, encoding='utf-8-sig')

#### 10개 카드사의 id만 따로 저장 (데이터 추가 추출 시 활용하기 위해서)

In [65]:
valid_ids = pd.DataFrame(top_df.loc[:, '카드번호'])

In [68]:
len(valid_ids)

2502

In [67]:
valid_ids.to_csv('data/card_id.csv', index=False, encoding='utf-8-sig')

#### 수정사항 있을 경우

In [76]:
# valid_ids = pd.read_csv('data/card_id.csv')
valid_list = valid_ids['카드번호'].tolist()

In [79]:
if __name__ == "__main__":
    # 1차 수집
    print("=== 1차 수집 시작 ===")
    first_success_data, first_fail_ids = collect_cards_batch(valid_list, desc="1차 수집")
    print(f"1차 완료: 성공 {len(first_success_data)}건, 실패 {len(first_fail_ids)}건")

    # 2. 실패한 건이 있다면 재수집 실행
    if first_fail_ids:
        print("\n=== 정보 요청 실패 건 재수집 시작 ===")
        # 실패한 ID 리스트를 그대로 다시 함수에 던져줍니다!
        retry_success_data, final_fail_ids = collect_cards_batch(first_fail_ids, desc="재수집")
        
        # 1차 성공 데이터와 재수집 성공 데이터를 합침
        final_card_data = first_success_data + retry_success_data
        print(f"재수집 완료: 복구 성공 {len(retry_success_data)}건, 최종 실패 {len(final_fail_ids)}건")
    else:
        final_card_data = first_success_data
        
    print(f"\n✅ 총 {len(final_card_data)}건의 카드 데이터 수집이 완료되었습니다.")

=== 1차 수집 시작 ===


1차 수집: 100%|████████████████████████████████████████████████████████████████████| 2502/2502 [33:46<00:00,  1.23it/s]

1차 완료: 성공 2502건, 실패 0건

✅ 총 2502건의 카드 데이터 수집이 완료되었습니다.


In [80]:
df = pd.DataFrame(final_card_data)
df.sort_values(by='카드번호', inplace=True)
df.tail()

,카드번호,카드명,카드사,카드타입,출시일,연회비,전월실적,카테고리,주요혜택,발급가능여부
0,1,신한카드 Hi-Point,신한카드,CRD,,8000,0,"쇼핑, 모든가맹점, 주유, 금융, 통신, 적립","전국가맹점 0.2~2.0% 적립, 특정가맹점 1~5% 적립, 에이치디현대오일뱅크 6...",X
1,2,신한카드 Love,신한카드,CRD,,8000,200000,"백화점, 패밀리레스토랑, 주유소, 영화, 적립, 테마파크, 경기관람","백화점/할인점 5% 할인, 스타벅스 20% 할인, GS칼텍스 60원/L 할인",X
2,3,신한카드 Lady,신한카드,CRD,,5000,300000,"쇼핑, 푸드, 영화, 주유소, 교육/육아, 테마파크, 통신, 경기관람","3대백화점 5% 할인, 패밀리레스토랑 20% 할인, 놀이공원 30~50% 할인",X
3,4,SK에너지 신한카드The you,신한카드,CRD,,7000,300000,"주유소, 대중교통, 무이자할부, 테마파크, 영화, 헤어, 경기관람, 여행사","SK주유소 100원/L 할인, 지하철,버스,택시 3~7% 할인, 놀이공원 30~50...",X
4,8,신한카드 The CLASSIC-Y,신한카드,CRD,,100000,0,"선택형, 모든가맹점, 주유소, 생활, 면세점, 진에어, 공연/전시, 프리미엄","Gift Option 매년1회 선택이용, 마이신한포인트 0.7~5% 적립, GS칼텍...",O


In [84]:
# df.to_csv('data/card_data.csv', index=False, encoding='utf-8-sig')